In [1]:
import torch
import torch_directml

# Detect all DirectML-capable GPUs (AMD/Intel on Windows)
print("Available DirectML devices:")
dml_device = None
for i in range(torch_directml.device_count()):
    name = torch_directml.device_name(i)
    print(f"  Device {i}: {name}")
    # Prefer the RX 7900 XT (dedicated GPU)
    if "7900" in name or ("RX" in name and "Radeon(TM) Graphics" not in name):
        dml_device = torch_directml.device(i)
        print(f"  -> Selected for inference")

# Fall back to device 0 if no dedicated GPU found by name filter
if dml_device is None:
    dml_device = torch_directml.device(torch_directml.device_count() - 1)
    print(f"  -> Falling back to last device: {torch_directml.device_name(torch_directml.device_count()-1)}")

print(f"\nUsing DirectML device handle: {dml_device}")


Available DirectML devices:
  Device 0: AMD Radeon(TM) Graphics 
  Device 1: AMD Radeon RX 7900 XT 
  -> Selected for inference

Using DirectML device handle: privateuseone:1


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Model Selection ---
# RX 7900 XT has 20 GB VRAM — flan-t5-xxl in float16 uses ~11 GB, so it fits.
# RTX 2080 Super has 8 GB VRAM  — use "google/flan-t5-xl" (~5.4 GB in float16).
MODEL_NAME = "google/flan-t5-xl"

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model in float16 (no quantization needed — DirectML handles AMD GPU natively)...")
# Load weights onto CPU first in float16 to keep peak RAM usage low,
# then we push the whole model to the DirectML device in one shot.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,         # ~11 GB for xxl — fits in 20 GB VRAM
    low_cpu_mem_usage=True,      # stream weights on-the-fly to reduce peak RAM
)

print(f"Moving model to DirectML device ({dml_device}) ...")
model = model.to(dml_device)
model.eval()
print("Model loaded and ready on AMD RX 7900 XT!")

# --- 10 Ensemble Instructions (from the paper) ---
instructions = [
    "Improve the search effectiveness by suggesting expansion terms for the query:",
    "Recommend expansion terms for the query to improve search results:",
    "Improve the search effectiveness by suggesting useful expansion terms for the query:",
    "Maximize search utility by suggesting relevant expansion phrases for the query:",
    "Enhance search efficiency by proposing valuable terms to expand the query:",
    "Elevate search performance by recommending relevant expansion phrases for the query:",
    "Boost the search accuracy by providing helpful expansion terms to enrich the query:",
    "Increase the search efficacy by offering beneficial expansion keywords for the query:",
    "Optimize search results by suggesting meaningful expansion terms to enhance the query:",
    "Enhance search outcomes by recommending beneficial expansion terms to supplement the query:"
]

def generate_ensemble_expansion(query):
    """Generates an ensemble of expansions for a given query using 10 prompts."""

    # Build the 10 prompts
    prompts = [f"{instruction} '{query}'" for instruction in instructions]

    # Tokenize as a batch and send to the DirectML device
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(dml_device)

    # Generate — parameters from the paper (Section 4)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,       # keep keywords concise
            do_sample=True,          # nucleus sampling
            top_p=0.92,
            top_k=200,
            repetition_penalty=1.2,
            num_return_sequences=1,
        )

    # Decode and de-duplicate
    generated_keywords = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_keywords = " ".join(generated_keywords)
    unique_keywords = list(set(all_keywords.lower().split()))
    return " ".join(unique_keywords)

# --- Quick smoke test ---
if __name__ == "__main__":
    test_query = "do goldfish grow"
    print(f"\nOriginal Query: {test_query}")
    expanded_terms = generate_ensemble_expansion(test_query)
    print(f"Ensemble Expanded Terms: {expanded_terms}")
    final_query = f"{test_query} {expanded_terms}"
    print(f"Final Reformulated Query: {final_query}")


Loading tokenizer for google/flan-t5-xl...
Loading model in float16 (no quantization needed — DirectML handles AMD GPU natively)...


Loading weights: 100%|██████████| 558/558 [00:04<00:00, 120.99it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Moving model to DirectML device (privateuseone:1) ...
Model loaded and ready on AMD RX 7900 XT!

Original Query: do goldfish grow


c:\Users\andre\OneDrive - Delft University of Technology\DSAIT4050InformationRetrieval\Group_Assignment\.venv\Lib\site-packages\transformers\models\t5\modeling_t5.py:473: UserWarning: The operator 'aten::clamp.Tensor_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  hidden_states = torch.clamp(hidden_states, min=-clamp_value, max=clamp_value)


Ensemble Expanded Terms: are growing, easily? be tropical longer old live do shoving, 12 grow fish up in large i foot a length years and goldfish can to
Final Reformulated Query: do goldfish grow are growing, easily? be tropical longer old live do shoving, 12 grow fish up in large i foot a length years and goldfish can to


In [4]:
other_queries = [
    "what is the capital of France",
    "how to bake a chocolate cake",
    "best practices for machine learning",
    "symptoms of the common cold",
    "history of the Roman Empire"
]

for query in other_queries:
    print(f"\nOriginal Query: {query}")
    
    expanded_terms = generate_ensemble_expansion(query)
    print(f"Ensemble Expanded Terms: {expanded_terms}")
    
    final_query = f"{query} {expanded_terms}"
    print(f"Final Reformulated Query: {final_query}")


Original Query: what is the capital of France
Ensemble Expanded Terms: name what the city france is of capital 2014 and arrondissement paris, france?
Final Reformulated Query: what is the capital of France name what the city france is of capital 2014 and arrondissement paris, france?

Original Query: how to bake a chocolate cake
Ensemble Expanded Terms: cake, chocolate for the holiday decorating how make in buttercream a skillet bake cake to
Final Reformulated Query: how to bake a chocolate cake cake, chocolate for the holiday decorating how make in buttercream a skillet bake cake to

Original Query: best practices for machine learning
Ensemble Expanded Terms: practices machine for training learning, methods of algorithm competitions learning in learning: practices, best
Final Reformulated Query: best practices for machine learning practices machine for training learning, methods of algorithm competitions learning in learning: practices, best

Original Query: symptoms of the common co